# Análise dos resultados da metaheurística Simulated Annealing (SA)

Resultados em `stochastic-results-sa/` no padrão:
`experiment-{alpha}-{temperature}-{temperature_max}-{alpha_sa}-{max_iters_sa}-{delta_type}-{first_improve}-{use_preprocessing}`

Objetivos: comparar solução inicial vs melhorada pelo SA; impacto do alpha (principalmente cenário 0); parâmetros que performaram melhor; **export LaTeX apenas da melhor combinação**.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.figsize"] = (8, 5)

from analysis_utils import (
    parse_sa_result,
    parse_experiment_folder_sa,
    parse_experiment_name_sa,
    export_sa_best_to_latex,
    instance_sort_key,
)

BASE_SA = Path("stochastic-results-sa")
if not BASE_SA.exists():
    raise FileNotFoundError(f"Pasta {BASE_SA} não encontrada. Execute os experimentos SA primeiro.")

In [ ]:
# Carregar todos os resultados SA
raw = parse_experiment_folder_sa(BASE_SA)
records = [r for _, r in raw]
df_all = pd.DataFrame(records)

# Parâmetros do nome do experimento
params = [parse_experiment_name_sa(r["experiment"]) for _, r in raw]
for key in (params[0].keys() if params else []):
    df_all[key] = [p.get(key) for p in params]

print(f"Total de arquivos: {len(df_all)}")
print(f"Experimentos únicos: {df_all['experiment'].nunique()}")
df_all.head()

## 1. Solução inicial vs SA: melhoria obtida

- **Start_UB**: valor da solução inicial.
- **LB**: valor da solução após SA (objetivo minimização, menor é melhor).
- **improvement** = Start_UB - LB (quanto o SA melhorou).

In [ ]:
df_all["improvement"] = df_all["Start_UB"] - df_all["LB"]
df_all["improvement_pct"] = np.where(
    df_all["Start_UB"] > 0,
    100 * df_all["improvement"] / df_all["Start_UB"],
    np.nan,
)

agg = df_all.groupby("experiment").agg(
    LB_medio=("LB", "mean"),
    Start_UB_medio=("Start_UB", "mean"),
    improvement_medio=("improvement", "mean"),
    improvement_pct_medio=("improvement_pct", "mean"),
    runtime_medio=("Runtime", "mean"),
    attended_s0_medio=("attended_s0", "mean"),
    n_instances=("instance_name", "nunique"),
).reset_index()
agg = agg.sort_values("LB_medio")
agg.head(15)

In [ ]:
# Melhor combinação de parâmetros: menor LB médio (melhor qualidade)
best_exp = agg.iloc[0]["experiment"]
print("Melhor experimento (menor LB médio):", best_exp)
print(agg.iloc[0])

## 2. Impacto do alpha na solução (primeiro estágio – cenário 0)

Blocos atendidos no cenário 0 e valor da solução (LB) por alpha.

In [ ]:
by_alpha = df_all.groupby("alpha").agg(
    LB_medio=("LB", "mean"),
    Start_UB_medio=("Start_UB", "mean"),
    improvement_medio=("improvement", "mean"),
    attended_s0_medio=("attended_s0", "mean"),
    attended_s0_std=("attended_s0", "std"),
).reset_index()
by_alpha

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax1, ax2 = axes
ax1.plot(by_alpha["alpha"], by_alpha["attended_s0_medio"], "o-", label="Blocos atendidos (s0)")
ax1.fill_between(
    by_alpha["alpha"],
    by_alpha["attended_s0_medio"] - by_alpha["attended_s0_std"].fillna(0),
    by_alpha["attended_s0_medio"] + by_alpha["attended_s0_std"].fillna(0),
    alpha=0.3,
)
ax1.set_xlabel("Alpha")
ax1.set_ylabel("Blocos atendidos (cenário 0)")
ax1.set_title("Alpha vs atendimento no primeiro estágio (SA)")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(by_alpha["alpha"], by_alpha["LB_medio"], "s-", color="C1", label="LB médio")
ax2.plot(by_alpha["alpha"], by_alpha["Start_UB_medio"], "^-", color="C2", label="Start UB médio")
ax2.set_xlabel("Alpha")
ax2.set_ylabel("Valor da solução")
ax2.set_title("Alpha vs qualidade da solução (SA)")
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Quais parâmetros performaram melhor?

Ranking por LB médio; influência de temperature, alpha_sa, max_iters_sa, delta_type, first_improve, use_preprocessing.

In [ ]:
# Top 20 combinações por LB médio
agg_sorted = agg.sort_values("LB_medio").head(20)
agg_sorted[["experiment", "LB_medio", "improvement_medio", "runtime_medio", "attended_s0_medio"]]

In [ ]:
# Impacto de cada parâmetro: média de LB por nível
for param in ["alpha", "temperature", "temperature_max", "alpha_sa", "max_iters_sa", "delta_type", "first_improve", "use_preprocessing"]:
    if param not in df_all.columns:
        continue
    effect = df_all.groupby(param)["LB"].mean().sort_values()
    print(f"\n{param}: (menor LB = melhor)")
    print(effect.to_string())

In [ ]:
# Boxplot: LB por alpha_sa e por max_iters_sa
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df_all.boxplot(column="LB", by="alpha_sa", ax=axes[0])
axes[0].set_title("LB por alpha_sa")
axes[0].set_xlabel("alpha_sa")
df_all.boxplot(column="LB", by="max_iters_sa", ax=axes[1])
axes[1].set_title("LB por max_iters_sa")
axes[1].set_xlabel("max_iters_sa")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 4. Exportação LaTeX (apenas melhor configuração)

Exportamos uma única tabela com os resultados da **melhor combinação de parâmetros** (menor LB médio).

In [ ]:
def export_best_sa_to_latex(df_full: pd.DataFrame, best_experiment: str, out_path: Path) -> str:
    df_best = df_full[df_full["experiment"] == best_experiment].copy()
    # Ordenar: cidade (alto-santo, limoeiro) -> tamanho (500, 1000, 2000) -> id (1, 2, 3, ...)
    df_best = df_best.sort_values("instance_name", key=lambda c: c.map(instance_sort_key))
    caption = f"Results of Simulated Annealing for stochastic instances (best config: {best_experiment})."
    label = "tab:sa-stochastic-best"
    latex = export_sa_best_to_latex(df_best, caption=caption, label=label, use_rowcolor=True)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(latex, encoding="utf-8")
    return latex

In [ ]:
OUT_LATEX_SA = Path("latex-tables-sa")
best_exp = agg.sort_values("LB_medio").iloc[0]["experiment"]
latex_str = export_best_sa_to_latex(df_all, best_exp, OUT_LATEX_SA / "table-sa-best.tex")
print(f"Melhor configuração: {best_exp}")
print(f"Tabela salva em {OUT_LATEX_SA / 'table-sa-best.tex'}")
print("\n--- Prévia LaTeX ---\n")
print(latex_str[:2000])